In [ ]:
import pandas as pd 
import numpy as np 
from IPython.display import display
import matplotlib.pyplot as plt 
from pathlib import Path
from xlsx2csv import Xlsx2csv

# Корень репозитория: каталог, где лежит data/ (cwd = репо или notebooks/)
_cwd = Path.cwd().resolve()
REPO_ROOT = _cwd if (_cwd / "data").is_dir() else _cwd.parent
DATA_DIR = REPO_ROOT / "data"

In [12]:
# Конвертация: .xlsx -> UTF-8 CSV (data/ и data/sales/)

def xlsx_to_csv(src: Path, dst: Path, *, sheetid: int | None = None) -> None:
    opts = dict(outputencoding="utf-8")
    if sheetid is not None:
        Xlsx2csv(str(src), **opts).convert(str(dst), sheetid=sheetid)
    else:
        Xlsx2csv(str(src), **opts).convert(str(dst))


# data/ — основной прайс, сделки, дополнительные срезы df_price_dataflat_*
# xlsx_to_csv(DATA_DIR / "df_price_dataflat_2020_2025.xlsx", DATA_DIR / "price_dataflat_2020_2025.csv")
# xlsx_to_csv(DATA_DIR / "df_deals_dataflat_2020_2025.xlsx", DATA_DIR / "deals_dataflat_2020_2025.csv")
for stem in (
    # "2023_01_06",
    # "2023_04_12",
    "2023_07_12",
    "2024_01_06",
    "2024_07_12",
    "2025_01_05",
):
    xlsx_to_csv(
        DATA_DIR / f"df_price_dataflat_{stem}.xlsx",
        DATA_DIR / f"price_dataflat_{stem}.csv",
    )

In [13]:
PRICE_CSV = DATA_DIR / "price_dataflat_2020_2025.csv"
DEALS_CSV = DATA_DIR / "deals_dataflat_2020_2025.csv"

PRICE_DATAFLAT_CSV = {
    "2020_2025": DATA_DIR / "price_dataflat_2020_2025.csv",
    "2023_01_06": DATA_DIR / "price_dataflat_2023_01_06.csv",
    "2023_04_12": DATA_DIR / "price_dataflat_2023_04_12.csv",
    "2023_07_12": DATA_DIR / "price_dataflat_2023_07_12.csv",
    "2024_01_06": DATA_DIR / "price_dataflat_2024_01_06.csv",
    "2024_07_12": DATA_DIR / "price_dataflat_2024_07_12.csv",
    "2025_01_05": DATA_DIR / "price_dataflat_2025_01_05.csv",
}

SAMPLE_N = 10000


def read_csv_sample(path, nrows=SAMPLE_N):
    opts = dict(encoding="utf-8", low_memory=False)
    if nrows is not None:
        opts["nrows"] = nrows
    return pd.read_csv(path, **opts)


price_parts = []
for name, path in PRICE_DATAFLAT_CSV.items():
    part = read_csv_sample(path, nrows=SAMPLE_N)
    part = part.assign(price_slice=name)
    price_parts.append(part)

price_df = pd.concat(price_parts, ignore_index=True)
deals_df = read_csv_sample(DEALS_CSV)


In [14]:
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)

In [17]:
price_df.shape

(70000, 41)

In [18]:
import numpy as np

def profile_columns(df: pd.DataFrame, table_name: str, n_examples: int = 2, random_state: int = 42):
    rng = np.random.default_rng(random_state)
    rows = []

    for col in df.columns:
        s = df[col]
        non_null = s.dropna()

        # unique count on the currently loaded dataframe (скорее всего это sample)
        unique_count = int(s.nunique(dropna=True))

        examples = []
        if len(non_null) > 0:
            # Если уникальных значений мало — покажем все.
            # Иначе покажем 2 случайных уникальных значения.
            try:
                uniq = non_null.drop_duplicates().to_numpy()
                if unique_count < 10:
                    picked = list(uniq)
                else:
                    if len(uniq) <= n_examples:
                        picked = list(uniq)
                    else:
                        idx = rng.choice(len(uniq), size=n_examples, replace=False)
                        picked = [uniq[i] for i in idx]
                examples = [repr(x) for x in picked]
            except Exception:
                # fallback: более простой выбор значений
                try:
                    if unique_count < 10:
                        picked = non_null.drop_duplicates().tolist()
                    else:
                        picked = non_null.sample(n=min(n_examples, len(non_null)), random_state=random_state).tolist()
                    examples = [repr(x) for x in picked]
                except Exception:
                    examples = [repr(non_null.iloc[i]) for i in range(min(n_examples, len(non_null)))]

        rows.append({
            'table': table_name,
            'column': col,
            'nunique': unique_count,
            'examples': examples,
        })

    return pd.DataFrame(rows)

profiles_price = profile_columns(price_df, 'price_df', n_examples=2, random_state=42)
profiles_deals = profile_columns(deals_df, 'deals_df', n_examples=2, random_state=42)

profiles = pd.concat([profiles_price, profiles_deals], ignore_index=True)
display(profiles)


,table,column,nunique,examples
0,price_df,pool,50808,"['1145@5078@1@34@6@61,7@2', '1735@17675@3@496@5@37,1@2']"
1,price_df,ID Проекта,251,"[np.int64(1349), np.int64(1348)]"
2,price_df,ID Проекта для окружения,251,"[np.int64(1705), np.int64(610)]"
3,price_df,ID Корпус,975,"[np.int64(5842), np.int64(8653)]"
4,price_df,Проект,251,"['Сердце столицы', 'Симферопольский']"
5,price_df,Девелопер,168,"['Русский монолит', 'Подольский ДСК']"
6,price_df,Класс проекта,4,"['бизнес', 'премиум', 'де-люкс', 'комфорт']"
7,price_df,Регион,3,"['Москва', 'Новая Москва', 'Московская область']"
8,price_df,Округ,39,"['Дмитров городской округ', 'Красногорск городской округ']"
9,price_df,Район,113,"['Пушкино (г.)', 'Первомайское']"


In [19]:
set(price_df.columns) & set(deals_df.columns)

{'ID Корпус',
 'ID Проекта',
 'lat',
 'lng',
 'Бюджет',
 'Бюджет без скидки',
 'Девелопер',
 'Класс проекта',
 'Ключи',
 'Комнатность',
 'Наличие бюджета',
 'Округ',
 'Площадь',
 'Проект',
 'Район',
 'Регион',
 'Секция',
 'Старт продаж',
 'Тип помещения',
 'Условный номер',
 'Цена',
 'Цена без скидки',
 'Этаж'}

## Что делать дальше
1. После этого EDA выберем: (а) **как формировать `demand`**, (б) **какая именно `price`** (с учётом скидки), (в) **на каком уровне агрегирования** (проект×месяц, сегмент×месяц и т.д.).
2. Затем подготовим датасет для эластичности (агрегация + расчёт целевой переменной и признаков), но **без обучения ML**, как вы и просили изначально.

Если в вашей среде чтение Excel сработает медленно, скажите — сделаю версию, которая читает только схему/первые строки и избегает тяжёлых операций.

## Описание данных и полей (все датасеты)

Ниже перечислены файлы из папок `data/` и `data/sales/`, которые используются/могут использоваться для построения датасета эластичности спроса на квартиры к цене (без обучения ML на этом этапе).

### 1) `data/price_dataflat_2020_2025.csv` (таблица `price`, 40 полей)
Срезы того же формата: `price_dataflat_2023_01_06.csv` … `price_dataflat_2025_01_05.csv` (из `df_price_dataflat_*.xlsx` через xlsx2csv).

Назначение: снапшоты **предложения/прайс-листов** по объектам/лотам в разрезе времени. Здесь есть цена предложения, параметры скидок/бюджета и признак экспозиции.

Ключи / идентификаторы:
- `pool`
- `ID Проекта`
- `ID Проекта для окружения`
- `ID Корпус`
- `Секция`
- `Условный номер`

Проект / девелопер / типология:
- `Проект`
- `Девелопер`
- `Класс проекта`
- `Тип помещения`
- `Тип кв/ап`
- `Комнатность`
- `Отделка в отчет`
- `Отделка К`
- `Отделка текст`

География (локализация):
- `Регион`
- `Округ`
- `Район`
- `lat`
- `lng`

Характеристики юнита:
- `Этаж`
- `Площадь`
- `Ключи`

Время / стадия / контекст отчёта:
- `Старт продаж`
- `Сдача К`
- `Договор К`
- `Стадия К`
- `Дата файла`
- `Начало/конец месяца`
- `Период`
- `Источник`

Цена / бюджет / скидки (признаки для эластичности цены):
- `Цена`
- `Цена без скидки`
- `Старый бюджет`
- `Бюджет`
- `Бюджет без скидки`
- `Скидка, %`
- `Изменение цены последнее`
- `Наличие бюджета`

Оффер / доступность:
- `Экспозиция`

Доп. признак:
- `Период` (возможный индикатор месяца/среза для привязки к сделкам)


### 2) `data/deals_dataflat_2020_2025.csv` (таблица `deals`, 53 поля; исходник `df_deals_dataflat_2020_2025.xlsx`)
Назначение: **факт сделок / регистраций** и связанных характеристик объекта, сделки, финансирования и цены. Это источник для построения `demand` (сколько сделок произошло) и факторов спроса.

Ключи / идентификаторы:
- `key`
- `ID Проекта`
- `ID Проекта для окруженияSales`
- `ID Корпус`
- `Секция`
- `Условный номер`

География:
- `Регион`
- `Округ`
- `Район`
- `Округ Направление`
- `lat`
- `lng`
- `Корпус`

Проект / девелопер / типология:
- `Проект`
- `Девелопер`
- `Класс проекта`
- `Тип помещения`
- `Комнатность`

Характеристики объекта:
- `Этаж`
- `Площадь`
- `Наличие бюджета`
- `Ключи`
- `Отделка`
- `Тип обременения`
- `Ипотека`
- `Стадия строительства`
- `Стадия строительства в дату ДДУ`
- `Срок сдачи`

Цена / бюджет в контексте сделки:
- `Цена`
- `Цена без скидки`
- `Бюджет`
- `Бюджет без скидки`
- `Цена ДДУ`
- `Цена кв. м`
- `Цена по ПД`
- `Бюджет по ПД`
- `Опт`

Сделка / финансы / юридические признаки (факторы, влияющие на спрос):
- `Покупатель ФЛ`
- `Покупатель ЮЛ`
- `Залогодержатель/Банк`
- `Тип сделки`
- `Тип обременения`
- `Тип сделки`
- `Продавец ЮЛ`
- `Продавец ФЛ ID`
- `Длительность обременения`
- `Номер регистрации`
- `Тип обременения`
- `Дата регистрации модель`

Время (для таргета demand):
- `Дата ДДУ`
- `Дата регистрации`
- `Дата регистра_`
- `Дата подписания`
- `Старт продаж`

Тексты / описания:
- `Описание помещения`

Доп. объектные/вспомогательные поля:
- `key` (уникальность строки)
- `ID дом.рф`
- `Покупатель ЮЛ`
- `Покупатель ФЛ`


### 3) `data/sales/` (доп. файлы: московский срез/справочники; в ноутбуке — CSV после конвертации)
Эти файлы выглядят как “исходники/промежуточные таблицы” по Москве на близкую дату (`24.12.09` и `2024.12.09`). Они полезны, чтобы понять, откуда берутся поля цены/скидок/объектов.

3.1) `data/sales/price_24.12.09_msk.csv` (таблица `price`, 62 поля)
Назначение: цена/бюджет/скидки **по лотам** и их динамика в прайс-листе.

Идентификаторы:
- `ID проекта`
- `ID корпуса`
- `ID лота`
- `Секция/Подъезд`
- `Условный номер`
- `Идентификатор во внешнем источнике`

Проект / география / локация:
- `Название ЖК`
- `Регион`
- `Город`
- `Административный округ`
- `Район`
- `Корпус`
- `Класс`
- `Конструктив`
- `Девелопер`
- `Застройщик`

Стадия / время:
- `Стадия строительной готовности`
- `Старт продаж`
- `Заявленная дата РВЭ`
- `Дата первого появления лота в прайсе`
- `Первое появление в прайс-листе`
- `Срок пребывания лота в прайсе на момент последнего исторического слоя, дней`
- `Дата сбора`
- `Источник данных`

Характеристики лота/юнита:
- `Этаж`
- `Номер на этаже`
- `Количество комнат согласно типологии`
- `Количество комнат в источнике`
- `Площадь`
- `Тип помещения`

Цена/бюджет в разрезе прайса:
- `Цена м2`
- `Бюджет`
- `Цена м2 без скидки`
- `Бюджет без скидки`
- `Цена м2 со скидкой`
- `Бюджет со скидкой`
- `Цена м2 с отделкой`
- `Бюджет с отделкой`
- `Примечание по скидкам`

Детали скидок/акций:
- `Включение скидок/акций в ценообразование`
- `Наличие скидки/акции на объект недвижимости`
- `Размер скидки на объект недвижимости`
- `Скидка м2, руб`
- `Скидка м2,%`
- `Описание скидки/акции`

Отделка:
- `Наличие отделки по объекту недвижимости`
- `Описание отделки`
- `Отделка по корпусу`
- `Соответствие евроформату`

Виды/ориентация (условные доп. признаки):
- `Вид из окон`
- `Стороны света лота`

Описания и рост динамики:
- `Примечание по строительной готовности`
- `Тип договора`
- `Кол-во м/м по проекту`
- `Тип паркинга по корпусу/проекту`
- `Проектная площадь лотов по корпусу`
- `Корпус: кол-во этажей min`
- `Корпус: кол-во этажей max`
- `Корпус: кол-во этажей min`
- `Цена за 1 кв.м. первого появления лота в прайсе`
- `Бюджет покупки первого появления лота в прайсе`
- `Рост цены за 1 кв.м с первого появления в прайсе`
- `Рост бюджета покупки с первого появления в прайсе`
- `Стадия строительной готовности`


3.2) `data/sales/pd_msk_sheet1.csv` и `pd_msk_sheet2.csv` (два листа `pd_msk.xlsx`; по 34 поля)
Назначение: “карточки” объектов/лотa с базовой информацией (проект, стадия, тип договора, объектные атрибуты) и связью с сделкой/регистрацией.

(Список полей идентичен для `sheet1` и `sheet2`.)

- `ID проекта`
- `ID корпуса`
- `Проект`
- `Локация`
- `Город`
- `Округ`
- `Район`
- `Адрес`
- `Класс`
- `Конструктив`
- `Девелопер`
- `Застройщик`
- `Старт продаж`
- `Плановая дата РВЭ`
- `Стадия реализации`
- `Секция`
- `Стадия строительства`
- `Тип договора`
- `ID лота`
- `Номер объекта в ПД`
- `Тип объекта недвижимости`
- `Тип в декларации`
- `Этаж`
- `Кол-во комнат`
- `Кол-во комнат по bnMAP.pro`
- `Общая проектная площадь с НЛП`
- `Участие в сделке`
- `Дата договора сделки`
- `Участие в оптовой сделке`
- `Номер регистрации`
- `Тип сделки`
- `Тип продавца`
- `Тип покупателя`
- `Код покупателя`


3.3) `data/sales/msk_deals_2024.12.09.csv` (таблица сделок/лотa, 61 поле)
Назначение: сделки в Москве + связанная цена, динамика экспозиции, финансирование и обременения.

Идентификаторы:
- `ID проекта`
- `ID корпуса`
- `ID лота`
- `Секция`

Проект/география:
- `Проект`
- `Локация`
- `Город`
- `Округ`
- `Район`
- `Адрес корпуса`
- `Класс`
- `Конструктив здания`
- `Девелопер`
- `Застройщик`

Стадия / время:
- `Стадия строительной готовности на дату договора`
- `Старт продаж`
- `Заявленный срок ввода в эксплуатацию`
- `Дата договора`
- `Месяц и год даты договора`
- `Квартал и год договора`
- `Год договора`
- `Дата регистрации`
- `Месяц и год даты регистрации`
- `Квартал и год регистрации`
- `Год регистрации`
- `Дата актуальности данных`
- `История участия объекта в сделках (ПДн)`
- `Дата регистрации материнской сделки`
- `Дата договора материнской сделки`

Сделка (спрос):
- `Тип сделки`
- `Кол-во покупателей`
- `Код покупателя`
- `Тип покупателя`
- `Тип продавца`

Объектные атрибуты:
- `Тип объекта`
- `Этаж`
- `Секция`
- `Количество комнат`
- `Количество комнат в прайс-листе, типология bnmap.pro`
- `Универсальная комнатность`
- `Номер объекта в составе ЕГРН`
- `Номер объекта в ПД`
- `Площадь согласно ЕГРН`
- `Площадь согласно ПД`
- `Отделка по корпусу`
- `Цена отделки`

Цена и бюджеты (центральные для эластичности):
- `Цена за кв. метр`
- `Метод определения цены`
- `Расчетный бюджет объекта`
- `Цена 1 кв.м первого появления в экспозиции`
- `Бюджет покупки первого появления в экспозиции`

Скидки/включение скидок:
- `Включение скидок/акций в ценообразование`
- `Скидки по дате договора`

Финансирование/ипотека/залог:
- `Тип ипотеки`
- `Залогодержатель/Банк`
- `Кол-во месяцев обременения`

Экспозиция / динамика предложения:
- `Участие объекта в оптовой сделке`
- `Дата первого появления в экспозиции`
- `Срок в экспозиции до момента сделки, дней`
- `Бюджет покупки первого появления в экспозиции`
- `Рост цены за 1 кв.м за период экспонирования`
- `Рост бюджета покупки за период экспонирования`

Повторные сделки:
- `Количество повторных сделок`

